# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Credit Card Fraud Detection** dari Kaggle.
Dataset ini berisi transaksi kartu kredit yang dilakukan oleh pemegang kartu Eropa pada bulan September 2013.
Dataset ini bersifat **sangat imbalanced** karena hanya 0.172% transaksi yang merupakan fraud.

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:
1. **Sumber Dataset**: Kaggle - Credit Card Fraud Detection
2. **Karakteristik Dataset**: 
   - 284,807 transaksi
   - 31 fitur (Time, V1-V28, Amount, Class)
   - Fitur V1-V28 merupakan hasil PCA transformation
   - Class: 0 = Normal, 1 = Fraud

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

In [ ]:
df = pd.read_csv('../namadataset_raw/creditcard.csv')
df.head()

In [ ]:
df.tail()

In [ ]:
print(f"Jumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")
print(f"\nNama kolom: {list(df.columns)}")

In [ ]:
df.info()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.
Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

## 4.1 Deskripsi Statistik

In [ ]:
df.describe()

## 4.2 Cek Missing Values

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values per Kolom:")
print(missing_values)
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

## 4.3 Cek Data Duplikat

In [ ]:
duplicates = df.duplicated().sum()
print(f"Jumlah data duplikat: {duplicates}")
print(f"Persentase duplikat: {duplicates / len(df) * 100:.2f}%")

## 4.4 Distribusi Kelas Target

In [ ]:
class_dist = df['Class'].value_counts()
print("Distribusi Class:")
print(class_dist)
print(f"\nFraud Percentage: {class_dist[1] / len(df) * 100:.4f}%")
print(f"Normal Percentage: {class_dist[0] / len(df) * 100:.4f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='Class', data=df, ax=axes[0])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Class (0: Normal, 1: Fraud)')
axes[0].set_ylabel('Count')

for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom')

class_pct = df['Class'].value_counts(normalize=True) * 100
axes[1].pie(class_pct, labels=['Normal', 'Fraud'], autopct='%1.4f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Class Distribution (Percentage)')

plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 Distribusi Fitur Amount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Amount'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Amount')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

sns.boxplot(x='Class', y='Amount', data=df, ax=axes[1])
axes[1].set_title('Amount by Class')
axes[1].set_xlabel('Class (0: Normal, 1: Fraud)')
axes[1].set_ylabel('Amount')

plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Amount - Mean: {df['Amount'].mean():.2f}, Std: {df['Amount'].std():.2f}")
print(f"Amount - Min: {df['Amount'].min():.2f}, Max: {df['Amount'].max():.2f}")

## 4.6 Distribusi Fitur Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Time'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Frequency')

sns.boxplot(x='Class', y='Time', data=df, ax=axes[1])
axes[1].set_title('Time by Class')
axes[1].set_xlabel('Class (0: Normal, 1: Fraud)')
axes[1].set_ylabel('Time (seconds)')

plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Distribusi Fitur V1-V28

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]

fig, axes = plt.subplots(7, 4, figsize=(20, 28))
axes = axes.flatten()

for i, col in enumerate(v_cols):
    sns.histplot(data=df, x=col, hue='Class', bins=30, ax=axes[i],
                 kde=True, stat='density', common_norm=False)
    axes[i].set_title(f'{col} Distribution', fontsize=10)
    axes[i].set_xlabel('')

plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_v_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.8 Korelasi Fitur

In [ ]:
corr_matrix = df.corr()

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
corr_with_class = df.corr()['Class'].drop('Class').sort_values(ascending=False)
print("Korelasi fitur dengan Class:")
print(corr_with_class)

fig, ax = plt.subplots(figsize=(12, 8))
corr_with_class.plot(kind='bar', ax=ax, color=['#e74c3c' if x < 0 else '#2ecc71' for x in corr_with_class])
ax.set_title('Feature Correlation with Class')
ax.set_xlabel('Features')
ax.set_ylabel('Correlation')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.9 Analisis Outlier

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(20, 28))
axes = axes.flatten()

for i, col in enumerate(v_cols):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(f'{col}', fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

plt.suptitle('Boxplot of V1-V28 Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('namadataset_preprocessing/eda_outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.10 Ringkasan EDA

Berdasarkan EDA yang telah dilakukan, berikut adalah temuan utama:

1. **Missing Values**: Tidak terdapat missing values dalam dataset
2. **Data Duplikat**: Terdapat data duplikat yang perlu dihapus
3. **Class Imbalance**: Dataset sangat imbalanced (0.17% fraud)
4. **Distribusi Fitur**: Fitur V1-V28 sudah di-scale (hasil PCA), namun Time dan Amount belum
5. **Korelasi**: Beberapa fitur memiliki korelasi moderat dengan Class
6. **Outlier**: Terdapat outlier pada beberapa fitur V, namun karena ini data transaksi fraud, outlier bisa jadi penting

Langkah preprocessing yang diperlukan:
- Hapus data duplikat
- Standarisasi fitur Time dan Amount
- Split data menjadi train dan test set dengan stratifikasi

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Berikut adalah tahapan-tahapan yang dilakukan:
1. Menghapus Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Train-Test Split

## 5.1 Menghapus Missing Values

In [ ]:
print(f"Jumlah baris sebelum hapus missing values: {len(df)}")
df = df.dropna()
print(f"Jumlah baris setelah hapus missing values: {len(df)}")

## 5.2 Menghapus Data Duplikat

In [ ]:
print(f"Jumlah baris sebelum hapus duplikat: {len(df)}")
df = df.drop_duplicates()
print(f"Jumlah baris setelah hapus duplikat: {len(df)}")

## 5.3 Standarisasi Fitur Time dan Amount

In [ ]:
scaler = StandardScaler()
df['Scaled_Time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df['Scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))

df.drop(['Time', 'Amount'], axis=1, inplace=True)

print("Kolom setelah standarisasi:")
print(df.columns.tolist())
df.head()

## 5.4 Train-Test Split

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Shape Training Data: {X_train.shape}")
print(f"Shape Testing Data: {X_test.shape}")
print(f"\nTraining - Normal: {(y_train == 0).sum()}, Fraud: {(y_train == 1).sum()}")
print(f"Testing - Normal: {(y_test == 0).sum()}, Fraud: {(y_test == 1).sum()}")

## 5.5 Simpan Data Preprocessing

In [ ]:
train_df = X_train.copy()
train_df['Class'] = y_train.values
train_df.to_csv('namadataset_preprocessing/creditcard_train_preprocessed.csv', index=False)

test_df = X_test.copy()
test_df['Class'] = y_test.values
test_df.to_csv('namadataset_preprocessing/creditcard_test_preprocessed.csv', index=False)

full_df = pd.concat([X_train, X_test], axis=0)
full_df['Class'] = pd.concat([y_train, y_test], axis=0)
full_df.to_csv('namadataset_preprocessing/creditcard_preprocessed.csv', index=False)

print("Data preprocessed berhasil disimpan!")
print(f"  - creditcard_train_preprocessed.csv ({len(train_df)} rows)")
print(f"  - creditcard_test_preprocessed.csv ({len(test_df)} rows)")
print(f"  - creditcard_preprocessed.csv ({len(full_df)} rows)")